In [50]:
# hoping to test out implementation with multiple classes here
from transformers import pipeline
import pandas as pd
import os
from scipy.special import softmax

os.chdir("/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies/Data")
data = pd.read_csv("MN_Health_short.csv")

# extracting text from data
text = data.pop("Text").str.slice(0,100)
lst = text.to_list()

hypothesis_template = "This text is about {}"
classes_verbalized = ["economic relief", "reopening", "jobs", "housing", "vaccines","testing", "positive cases", 
                      "healthcare professionals", "healthcare infrastructure", "other", "research", "food"]

zeroshot_classifier = pipeline("zero-shot-classification", 
                               model="mlburnham/Political_DEBATE_DeBERTa_large_v1.1", 
                               device = "mps")  # change the model identifier here

output = zeroshot_classifier(lst, classes_verbalized, hypothesis_template=hypothesis_template, multi_label=True)

clean_output = []

for dct in output:
    nd = {}
    nd["sequence"] = dct["sequence"]
    for idx, label in enumerate(dct["labels"]):
        nd[label] = dct["scores"][idx]
    clean_output.append(nd)

df = pd.DataFrame(clean_output)

# mask to identify the columns with probabilities
prob_cols = df.columns.difference(['sequence'])
# creating a copy to apply softmax to
df2 = df.copy(deep=False) 
# Applying softmax to the copy
df2[prob_cols] = softmax(df[prob_cols].values, axis=1)

# combining probabilities with the original data
output =pd.concat([data, df2], axis=1)

df.to_csv('MAtopics_ZS_Burnham_modified_prompt_100_500chr.csv')



Device set to use mps


In [53]:
type(output.iloc[3,])

pandas.core.series.Series

In [2]:
df

num_cols = df.columns.difference(['sequence'])
df2 = df.copy(deep=False)  
df2[num_cols] = softmax(df[num_cols].values, axis=1)
df2

,sequence,covid,other
0,The Minnesota Department of Health today annou...,0.999999,1.050766e-06
1,The Minnesota Department of Health (MDH) today...,1.000000,5.361989e-07
2,This case marks the first documented instance ...,0.445633,5.543672e-01
3,A rapidly growing outbreak of variant COVID-19...,0.999999,1.202849e-06
4,As health officials continue to report more ca...,0.999999,1.416599e-06
5,Reflecting ongoing concerns about the potentia...,0.026314,9.736865e-01
6,May 12 will be the last day of testing at the ...,0.999999,1.086696e-06
7,Participating health plans starting this work ...,0.309488,6.905116e-01
8,The Minnesota Department of Health (MDH) today...,0.999992,7.737041e-06


In [58]:
softmax(11*[0]+[1])

array([0.07289543, 0.07289543, 0.07289543, 0.07289543, 0.07289543,
       0.07289543, 0.07289543, 0.07289543, 0.07289543, 0.07289543,
       0.07289543, 0.19815031])